# Day 2 — Step 3: 이동평균선 계산 [필수 과제 — 문제2]

02_preprocessing에서 저장한 데이터를 불러와  
종목별 5일 이동평균(ma5)을 계산하고 결측치를 처리합니다.

---

## 이동평균선(Moving Average, MA)이란?

**최근 N일간의 종가를 매일 평균 낸 값**을 선으로 이어 그린 것입니다.  
하루하루 가격의 출렁임을 부드럽게 만들어서 **전체적인 방향(추세)** 을 보여줍니다.

```
MA5  (5일 평균)  → 단기 추세 (약 1주일)
MA20 (20일 평균) → 중기 추세 (약 1달)
MA60 (60일 평균) → 장기 추세 (약 3달)
```

이번 과제에서는 **MA5(5일 이동평균)** 만 계산합니다.

### 계산 방법 예시

```
날짜       종가(close)    MA5 계산 과정
------------------------------------------------------
1월 1일    100,000원      → 데이터 5일치 없음 → NaN
1월 2일    102,000원      → 데이터 5일치 없음 → NaN
1월 3일     98,000원      → 데이터 5일치 없음 → NaN
1월 4일    105,000원      → 데이터 5일치 없음 → NaN
1월 5일    103,000원      → (100+102+98+105+103)÷5 = 101,600원
1월 6일    107,000원      → (102+98+105+103+107)÷5 = 103,000원
```

> 처음 4일은 5일치 데이터가 없어서 계산이 안 됩니다.  
> 이렇게 계산 불가한 빈칸을 **NaN(Not a Number)** 이라고 합니다.

---

### 왜 NaN을 close 값으로 채우나요?

NaN을 그냥 두면 차트에 **구멍**이 생기고, 나중에 DB 저장 시 오류가 날 수 있습니다.  
0으로 채우면 차트에서 가격이 갑자기 0원으로 떨어지는 현상이 생깁니다.  

> → 이동평균이 없는 초기 구간은 **그날의 종가(close)로 대체**합니다.  
> 종가는 실제 거래된 가격이니까, 가장 자연스러운 대체값입니다.

---

### 종목별로 따로 계산해야 하는 이유

```
여러 종목이 하나의 DataFrame에 세로로 합쳐져 있습니다:

  ...  KRW-BTC  2026-01-30  90,000,000  ← BTC 마지막 날
       KRW-ETH  2026-01-01   3,100,000  ← ETH 첫째 날 ⚠️ 종목이 바뀜!

→ BTC 90,000,000원과 ETH 3,100,000원을 섞어서 평균 내면 의미 없는 숫자!
→ ticker별로 나눠서 각각 이동평균을 계산해야 합니다
```

---
## 0. 라이브러리 불러오기

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

---
## 1. 데이터 불러오기

In [ ]:
df = pd.read_csv('ohlcv_preprocessed.csv', parse_dates=['date'])

print(f"데이터 로드 완료: {len(df)}행")
print(df.head())

---
## 2. [필수 과제 — 문제2] 5일 이동평균선 계산 및 결측치 처리

In [ ]:
def add_moving_average(df):
    """
    종목(ticker)별로 5일 이동평균(ma5)을 계산하고 NaN을 close로 대체합니다.

    처리 순서:
        1. groupby('ticker')로 종목별 분리
        2. 각 종목 내에서 close 기준 5일 rolling mean 계산
        3. NaN → 해당 행의 close 값으로 대체

    매개변수(Parameter):
        df : ticker, date, close 컬럼이 있는 DataFrame

    반환값(Return):
        ma5 컬럼이 추가된 DataFrame
    """
    df = df.copy()

    # ── 1단계: 종목별 5일 이동평균 계산 ──────────────────────────
    # groupby('ticker')   : 같은 ticker끼리 묶어서 처리
    # ['close']           : close 컬럼을 선택
    # .transform(lambda x: ...)
    #   → transform : 그룹별로 함수를 적용하되, 결과를 원래 DataFrame과 같은 크기로 맞춤
    #   → lambda x  : x라는 이름으로 각 종목의 close 시리즈를 받음
    #   → x.rolling(5).mean()
    #       rolling(5) : 최근 5행을 묶어서 처리하는 슬라이딩 윈도우
    #                    마치 창문을 하루씩 밀면서 창문 안 5일의 평균을 구하는 방식
    #       .mean()    : 묶인 5개 값의 평균
    df['ma5'] = df.groupby('ticker')['close'].transform(
        lambda x: x.rolling(5).mean()
    )

    # ── 2단계: NaN을 close 값으로 대체 ───────────────────────────
    # fillna() : NaN(빈값)을 특정 값으로 채우는 함수
    # df['close'] : NaN인 자리에 같은 행의 close 값을 넣겠다는 의미
    # 예) 1월 1일 ma5=NaN, close=100,000 → ma5=100,000 으로 대체됨
    df['ma5'] = df['ma5'].fillna(df['close'])

    return df

In [ ]:
# ── 함수 적용 ──────────────────────────────────────────────────────
df = add_moving_average(df)

# ── NaN이 남아있는지 검증 ──────────────────────────────────────────
# .isnull()    : NaN이면 True, 아니면 False
# .sum()       : True(=1)의 개수를 더함 → 전체 NaN 개수
nan_count = df['ma5'].isnull().sum()

if nan_count == 0:
    print("✅ NaN 없음: 결측치 처리 완료!")
else:
    print(f"⚠️  NaN이 {nan_count}개 남아있습니다. 처리 로직을 확인하세요.")

In [ ]:
# ── 종목별 close vs ma5 비교 출력 ─────────────────────────────────
print("[종목별 close와 ma5 비교 (앞 7일)]")
print()

for ticker in df['ticker'].unique():
    # df['ticker'] == ticker 조건이 True인 행만 추출
    ticker_df = df[df['ticker'] == ticker][['date', 'close', 'ma5']].head(7)
    print(f"--- {ticker} ---")
    print(ticker_df.to_string(index=False))
    print()

---
## 3. 전처리 완료 — 컬럼 순서 정리

모든 컬럼이 추가된 최종 DataFrame의 컬럼 순서를 보기 좋게 정돈합니다.

In [ ]:
# ── 최종 컬럼 순서 정의 ────────────────────────────────────────────
final_columns = [
    'ticker',             # 종목 코드
    'date',               # 날짜
    'open',               # 시가
    'high',               # 고가
    'low',                # 저가
    'close',              # 종가 (분석 기준)
    'volume',             # 거래량
    'price_change',       # 파생 지표: 가격 변동
    'price_change_pct',   # 파생 지표: 변동률(%)
    'high_low_diff',      # 파생 지표: 일중 변동폭
    'ma5',                # 파생 지표: 5일 이동평균
]
df = df[final_columns]

print("[최종 DataFrame 컬럼 순서]")
print(list(df.columns))
print()
print("[최종 데이터 미리보기]")
print(df.head())

In [ ]:
# ── 다음 단계로 전달 ──────────────────────────────────────────────
df.to_csv('ohlcv_final.csv', index=False)
print("ohlcv_final.csv 저장 완료 → 04_visualization.ipynb에서 이어서 작업합니다")